# Spotify Popularity Prediction Part 3

## Spark Session

In [1]:
from pyspark.sql import SparkSession

# Initialize Spark session with enough memory for the 189GB dataset
# 7 executors x 25g = 175g total executor memory across the cluster

spark = (
    SparkSession.builder
    .appName("Spotify_Part3_RF")
    .config("spark.driver.memory", "10g")
    .config("spark.executor.instances", "7")
    .config("spark.executor.memory", "25g")
    .config("spark.executor.memoryOverhead", "4g") 
    .config("spark.sql.shuffle.partitions", "100")  #reduced from 200 for sample
    .getOrCreate()
)

## Load Parquet Tables

In [2]:
from pyspark.sql import functions as F

data_path = "/expanse/lustre/projects/uci157/darenas/shared/spotify_clean_parquet"

# Load all tables
tracks = spark.read.parquet(data_path + "/tracks.parquet")
artists = spark.read.parquet(data_path + "/artists.parquet")
albums = spark.read.parquet(data_path + "/albums.parquet")
track_artists = spark.read.parquet(data_path + "/track_artists.parquet")

# Audio features
audio = spark.read.parquet("/expanse/lustre/projects/uci157/darenas/shared/spotify_clean_audio_features_parquet")

print("Data loaded")

Data loaded


## Cleaning

In [3]:
# Audio features cleaning
# Rename duration_ms to avoid confusion with tracks.duration_ms
audio = audio.withColumnRenamed("duration_ms", "audio_duration_ms")

# Drop rows with null response
audio = audio.filter(F.col("null_response").isNull())

# Cast all audio feature columns to double for ML compatibility
audio_cols = [
    "audio_duration_ms", "time_signature", "tempo",
    "key", "mode", "danceability", "energy",
    "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence"
]
for c in audio_cols:
    audio = audio.withColumn(c, F.col(c).cast("double"))

In [4]:
# Artist deduplication
# Some artists appear multiple times (ex. BTS has entries for each member's solo work)
# Keep only the row with the highest followers_total per artist name

from pyspark.sql.window import Window

artists_clean = artists.select(
    F.col("rowid").alias("artist_rowid"),
    F.col("id"),
    F.col("name"),
    F.col("followers_total"),
    F.col("popularity").alias("artist_popularity")
)

# Window: for each artist name, rank by followers descending
w = Window.partitionBy("name").orderBy(
    F.desc("followers_total"),
    F.desc("artist_popularity")
)
artists_clean = (
    artists_clean
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") == 1)  # keep only the top row per artist name
    .drop("rank")
)

In [5]:
# Only take cols we need from each

# Tracks
tracks_clean = tracks.select(
    F.col("rowid").alias("track_rowid"),
    F.col("id").alias("track_id"),
    F.col("name").alias("track_name"),
    F.col("popularity"),           # this is our prediction target
    F.col("duration_ms").alias("track_duration"),
    F.col("explicit"),
    F.col("album_rowid")
)

# Sample before joins
# Sampling on tracks first = all 4 joins run on 1.25M rows instead of 256M
# This prevents shuffle files from filling up the executor local disk to prevent spill error
tracks_clean = tracks_clean.sample(False, 0.005, seed=42)

# Albums
albums_clean = albums.select(
    F.col("rowid").alias("album_rowid"),
    F.col("popularity").alias("album_popularity"),
    "label",
    "total_tracks"
)


## Join Pipeline

In [6]:
# Join tracks to audio features, track_artists bridge table, artists, and albums
# All joins are left so we keep tracks even if audio/artist/album data is missing

main_df = (
    tracks_clean
    # Join audio features on track_id
    .join(audio, tracks_clean.track_id == audio.track_id, "left")
    .drop(audio["track_id"])                          # drop duplicate track_id from audio
    # Join bridge table to get artist_rowid per track
    .join(track_artists, tracks_clean.track_rowid == track_artists.track_rowid, "left")
    .drop(track_artists["track_rowid"])               # drop duplicate track_rowid
    # Join artists on artist_rowid
    .join(artists_clean, track_artists.artist_rowid == artists_clean.artist_rowid, "left")
    .drop(track_artists["artist_rowid"]).drop(artists_clean["artist_rowid"])
    # Join albums on album_rowid
    .join(albums_clean, tracks_clean.album_rowid == albums_clean.album_rowid, "left")
    .drop(albums_clean["album_rowid"])                # drop duplicate album_rowid
)

In [7]:
# Deduplicate tracks
# A track with multiple artists creates multiple rows after the join
# To fix this collapse to one row per track_id using agg

df_clean = main_df.groupBy("track_id").agg(
    F.first("track_name").alias("track_name"),
    F.first("name").alias("artist_name"),             # primary artist name
    F.first("popularity").alias("popularity"),         # track popularity (our target)
    F.max("followers_total").alias("followers_total"), # take the most-followed artist
    F.max("artist_popularity").alias("artist_popularity"),
    F.first("album_popularity").alias("album_popularity"),
    F.first("total_tracks").alias("total_tracks"),
    F.first("track_duration").alias("track_duration"),
    F.first("explicit").alias("explicit"),
    *[F.first(c).alias(c) for c in audio_cols]        # all audio features
)

# Drop rows missing the target or track id
df_clean = df_clean.na.drop(subset=["popularity", "track_id"])

## Preprocessing

In [8]:
# Feature engineering
# log1p(followers) compresses the huge range of follower counts (0 to 80M+)

df_clean = df_clean.withColumn("log_followers", F.log1p(F.col("followers_total")))

# Interaction feature: energy * danceability captures "hype" tracks
df_clean = df_clean.withColumn("energy_dance", F.col("energy") * F.col("danceability"))

In [9]:
# Imputer for missing values

from pyspark.ml.feature import Imputer
from pyspark.sql import Row

toy_data = spark.createDataFrame([
    Row(followers_total=1000.0,     tempo=120.0, danceability=None, energy=0.80),
    Row(followers_total=None,       tempo=130.0, danceability=0.65, energy=None),
    Row(followers_total=500000.0,   tempo=None,  danceability=0.55, energy=0.70),
    Row(followers_total=78000000.0, tempo=110.0, danceability=0.75, energy=0.90),
])

demo_cols = ["followers_total", "tempo", "danceability", "energy"]
imputer_demo = Imputer(
    inputCols=demo_cols,
    outputCols=[c + "_imp" for c in demo_cols]
).setStrategy("mean")  # replace nulls with column mean

imputer_demo.fit(toy_data).transform(toy_data).show(truncate=False)

+---------------+-----+------------+------+-------------------+---------+----------------+------------------+
|followers_total|tempo|danceability|energy|followers_total_imp|tempo_imp|danceability_imp|energy_imp        |
+---------------+-----+------------+------+-------------------+---------+----------------+------------------+
|1000.0         |120.0|NULL        |0.8   |1000.0             |120.0    |0.65            |0.8               |
|NULL           |130.0|0.65        |NULL  |2.6167E7           |130.0    |0.65            |0.7999999999999999|
|500000.0       |NULL |0.55        |0.7   |500000.0           |120.0    |0.55            |0.7               |
|7.8E7          |110.0|0.75        |0.9   |7.8E7              |110.0    |0.75            |0.9               |
+---------------+-----+------------+------+-------------------+---------+----------------+------------------+



In [10]:
# Handle missing values on real data
# Fill nulls with 0 for all numeric feature columns
# missing audio features = track has no audio data, fill as 0

numeric_cols_full = [
    "followers_total", "artist_popularity", "album_popularity", "total_tracks",
    "track_duration", "explicit", "log_followers", "energy_dance"
] + audio_cols

df_clean = df_clean.fillna(0, subset=numeric_cols_full)

In [11]:
from pyspark.ml.feature import VectorAssembler, Normalizer

# All numeric features 
feature_cols = [
    "followers_total", "artist_popularity", "album_popularity", "total_tracks",
    "track_duration", "explicit", "log_followers", "energy_dance"
] + audio_cols

# VectorAssembler combines all feature columns into a single vector column
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw",
    handleInvalid="keep"  # keep rows with remaining nulls rather than dropping
)

# Normalizer scales each row's feature vector to unit norm (L2)
normalizer = Normalizer(inputCol="features_raw", outputCol="features", p=2)

# Apply assembler and normalizer
df_assembled = assembler.transform(df_clean)
df_final = normalizer.transform(df_assembled).select(
    "track_id", "track_name", "artist_name", "popularity", "features"
)

# Cache so train/val/test splits don't recompute the full join and clean process
df_final = df_final.cache()
row_count = df_final.count()  # triggers caching
print(f"Preprocessed rows: {row_count:,}")
df_final.show(5, truncate=True)

Preprocessed rows: 1,278,282
+--------------------+--------------------+--------------------+----------+--------------------+
|            track_id|          track_name|         artist_name|popularity|            features|
+--------------------+--------------------+--------------------+----------+--------------------+
|0000RM7QYBRAsuFAQ...|             Mo n Me|           Modd Jobs|         0|(21,[0,3,4,6],[5....|
|0001qtVaca5vVqHv0...|               Arena|         Kronnospace|         0|(21,[0,1,3,4,6],[...|
|000BJtGc0SLMN5ixU...|   Trịnh Vương Maria|          Thanh Hoài|         0|(21,[0,1,2,3,4,6]...|
|000Wdim8xmdzFdAds...|Emotional Intelli...|Wave Particle Sin...|         0|(21,[0,1,3,4,6],[...|
|000XiFdhd8fypmIvU...|       Escaped Pulse|      Spiritual Yoga|         0|(21,[0,1,3,4,6],[...|
+--------------------+--------------------+--------------------+----------+--------------------+
only showing top 5 rows



## Train / Validation / Test Split

In [12]:
# 70% train, 15% validation, 15% test

train_df, val_df, test_df = df_final.randomSplit([0.7, 0.15, 0.15], seed=42)

print(f"Train:      {train_df.count():,}")
print(f"Validation: {val_df.count():,}")
print(f"Test:       {test_df.count():,}")

Train:      895,418
Validation: 191,376
Test:       191,488


## Distributed Random Forest: Model 1

In [13]:
from pyspark.ml.regression import RandomForestRegressor

# Model 1: baseline hyperparameters
# Using regression since popularity is a continuous 0-100 score

rf = RandomForestRegressor(
    labelCol="popularity",
    featuresCol="features",
    numTrees=50,    
    maxDepth=8,  
    featureSubsetStrategy="auto",
    seed=42
)

print("Fitting Model 1 (numTrees=50, maxDepth=8)")
rf_model = rf.fit(train_df)
print("Done")

Fitting Model 1 (numTrees=50, maxDepth=8)
Done


## Evaluation (Train vs Validation vs Test)

In [14]:
from pyspark.ml.evaluation import RegressionEvaluator

eval_rmse = RegressionEvaluator(labelCol="popularity", predictionCol="prediction", metricName="rmse")
eval_r2   = RegressionEvaluator(labelCol="popularity", predictionCol="prediction", metricName="r2")

# Predict on all 3 splits
pred_train = rf_model.transform(train_df)
pred_val   = rf_model.transform(val_df)
pred_test  = rf_model.transform(test_df)

print("Model 1: numTrees=50, maxDepth=8")
print(f"RMSE  train={eval_rmse.evaluate(pred_train):.3f}  val={eval_rmse.evaluate(pred_val):.3f}  test={eval_rmse.evaluate(pred_test):.3f}")
print(f"R2    train={eval_r2.evaluate(pred_train):.3f}    val={eval_r2.evaluate(pred_val):.3f}    test={eval_r2.evaluate(pred_test):.3f}")

Model 1: numTrees=50, maxDepth=8
RMSE  train=2.709  val=2.757  test=2.722
R2    train=0.708    val=0.707    test=0.708


In [15]:
cols_to_show = ["track_name", "artist_name", "popularity", "prediction"]

print("TRAIN examples")
pred_train.select(*cols_to_show).orderBy("track_name").show(10, truncate=False)

print("VALIDATION examples")
pred_val.select(*cols_to_show).orderBy("track_name").show(10, truncate=False)

print("TEST examples")
pred_test.select(*cols_to_show).orderBy("track_name").show(10, truncate=False)

TRAIN examples
+----------+---------------+----------+-------------------+
|track_name|artist_name    |popularity|prediction         |
+----------+---------------+----------+-------------------+
|          |Various Artists|0         |1.1987800680647311 |
|          |Various Artists|0         |1.096143831089155  |
|          |Various Artists|0         |1.096143831089155  |
|          |Various Artists|0         |0.6043595953184664 |
|          |Various Artists|0         |0.5423023018458281 |
|          |Various Artists|0         |0.4512086752807267 |
|          |Various Artists|0         |0.4512086752807267 |
|          |Various Artists|0         |0.37898768299582253|
|          |Various Artists|0         |1.096143831089155  |
|          |Various Artists|0         |0.502111287234105  |
+----------+---------------+----------+-------------------+
only showing top 10 rows

VALIDATION examples
+----------+---------------+----------+-------------------+
|track_name|artist_name    |popularity|

## Hyperparameter Comparison: Model 2

In [16]:
# Model 2: deeper trees and more trees to test for overfitting

rf2 = RandomForestRegressor(
    labelCol="popularity",
    featuresCol="features",
    numTrees=100,
    maxDepth=12,
    featureSubsetStrategy="auto",
    seed=42
)

print("Fitting Model 2 (numTrees=100, maxDepth=12)")
rf_model2 = rf2.fit(train_df)
print("Done")

pred_train2 = rf_model2.transform(train_df)
pred_val2   = rf_model2.transform(val_df)
pred_test2  = rf_model2.transform(test_df)

print("Model 2: numTrees=100, maxDepth=12")
print(f"RMSE  train={eval_rmse.evaluate(pred_train2):.3f}  val={eval_rmse.evaluate(pred_val2):.3f}  test={eval_rmse.evaluate(pred_test2):.3f}")
print(f"R2    train={eval_r2.evaluate(pred_train2):.3f}    val={eval_r2.evaluate(pred_val2):.3f}    test={eval_r2.evaluate(pred_test2):.3f}")

Fitting Model 2 (numTrees=100, maxDepth=12)
Done
Model 2: numTrees=100, maxDepth=12
RMSE  train=2.504  val=2.587  test=2.560
R2    train=0.751    val=0.742    test=0.742


## Feature Importance

In [17]:
# Feature importances from Model 1
# Higher value = more useful for predicting popularity

import pandas as pd

fi_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.featureImportances.toArray()
}).sort_values("importance", ascending=False)

print("Feature importances (Model 1):")
print(fi_df.to_string(index=False))

Feature importances (Model 1):
          feature  importance
 album_popularity    0.620257
     total_tracks    0.213915
   track_duration    0.062000
  followers_total    0.048010
artist_popularity    0.029234
    log_followers    0.022796
         explicit    0.003787
           energy    0.000000
         liveness    0.000000
 instrumentalness    0.000000
     acousticness    0.000000
      speechiness    0.000000
         loudness    0.000000
            tempo    0.000000
     danceability    0.000000
             mode    0.000000
              key    0.000000
   time_signature    0.000000
audio_duration_ms    0.000000
     energy_dance    0.000000
          valence    0.000000
